## 1) Install and Configure Environment



In [30]:
# Remove incompatible torchao if it is present in the runtime.
%pip uninstall -y torchao

# Install dependencies (run once in Colab)
%pip install -q --upgrade transformers accelerate peft datasets sentencepiece torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

# Basic GPU check
import torch
print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('Device name:', torch.cuda.get_device_name(0))

torch version: 2.10.0+cu128
CUDA available: False
CUDA device count: 0


## 2) Mount Google Drive (optional)

Use this to persist saved LoRA adapters and checkpoints across Colab sessions.

In [31]:
# Mount Google Drive (optional)
try:
    import importlib
    colab = importlib.import_module('google.colab')
    drive = colab.drive
    _ = drive
    DRIVE_AVAILABLE = True
    print('Colab environment detected; you can mount Drive if needed.')
    # drive.mount('/content/drive')  # uncomment to mount
except Exception:
    DRIVE_AVAILABLE = False
    print('Not running in Colab or google.colab not available.')

Colab environment detected; you can mount Drive if needed.


## 3) Clone / Load VulBERTa and Dependencies

Set `MODEL_NAME` to your VulBERTa HF repo or local path. The example below uses a placeholder; replace it with the actual model name.

In [32]:
# Set VulBERTa model name; replace with your specific VulBERTa repo path if needed
import os

# VulBERTa tokenizer options:
# 1. If you have a local VulBERTa checkpoint, set: MODEL_NAME = '/path/to/vulberta'
# 2. Or use a VulBERTa repo on HuggingFace (if available): MODEL_NAME = 'username/vulberta'
# 3. Default fallback: 'roberta-base' (VulBERTa is RoBERTa-based, so same tokenizer structure)
MODEL_NAME = os.environ.get('VULBERTA_MODEL', 'roberta-base')

NUM_LABELS = 2
MAX_LENGTH = 128

from transformers import AutoTokenizer, AutoModelForSequenceClassification

print(f"Loading VulBERTa/RoBERTa tokenizer and model from: '{MODEL_NAME}'")
print("Note: VulBERTa uses RoBERTa tokenizer and architecture.")
print("To use your custom VulBERTa model:")
print("  - Set env var: export VULBERTA_MODEL='/path/to/model'")
print("  - Or edit MODEL_NAME directly in this cell.")
print()

try:
    # Load VulBERTa tokenizer (RoBERTa-based)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
    model.config.label2id = {str(i): i for i in range(NUM_LABELS)}
    model.config.id2label = {i: str(i) for i in range(NUM_LABELS)}
    print(f'✓ Model and tokenizer loaded successfully.')
    print(f'  Model: {MODEL_NAME}')
    print(f'  Tokenizer vocab size: {len(tokenizer)}')
except Exception as e:
    print(f'Error loading model: {e}')
    print("Falling back to 'roberta-base' for demo purposes.")
    MODEL_NAME = 'roberta-base'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
    model.config.label2id = {str(i): i for i in range(NUM_LABELS)}
    model.config.id2label = {i: str(i) for i in range(NUM_LABELS)}
    print('✓ Fallback model (roberta-base) loaded.')

Loading VulBERTa/RoBERTa tokenizer and model from: 'roberta-base'
Note: VulBERTa uses RoBERTa tokenizer and architecture.
To use your custom VulBERTa model:
  - Set env var: export VULBERTA_MODEL='/path/to/model'
  - Or edit MODEL_NAME directly in this cell.



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model and tokenizer loaded successfully.
  Model: roberta-base
  Tokenizer vocab size: 50265


## 4) Prepare Vulnerability Dataset (tiny demo)

We create a tiny synthetic dataset for demonstration. Replace with your real vulnerability CSV/JSON as needed.

In [33]:
from datasets import Dataset

# Tiny example data (replace with your dataset)
examples = {
    'text': [
        'This function contains a use-after-free vulnerability.',
        'Safe code that parses input with validation.',
        'SQL injection via string concatenation in DB query.',
        'A benign utility function with no issues.'
    ],
    'label': [1, 0, 1, 0]
}

dataset = Dataset.from_dict(examples)
train_test = dataset.train_test_split(test_size=0.5, seed=42)
train_ds = train_test['train']
eval_ds = train_test['test']
print('Train size:', len(train_ds), 'Eval size:', len(eval_ds))

Train size: 2 Eval size: 2


## 5) Tokenizer and Data Collator

Tokenize texts and prepare data collator for padding.

In [34]:
def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_fn, batched=True)
eval_ds = eval_ds.map(tokenize_fn, batched=True)

# Keep only needed columns
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in ('input_ids','attention_mask','label')])
eval_ds = eval_ds.remove_columns([c for c in eval_ds.column_names if c not in ('input_ids','attention_mask','label')])

from transformers import DataCollatorWithPadding
collator = DataCollatorWithPadding(tokenizer)

print('Sample tokenized input:', train_ds[0]['input_ids'][:10])

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Sample tokenized input: [0, 32356, 3260, 14, 28564, 293, 8135, 19, 26567, 4]


## 6) Implement LoRA Modules and Integration

We use PEFT to add LoRA adapters. Adjust `target_modules` if necessary for your model architecture.

In [35]:
from peft import LoraConfig, get_peft_model, TaskType
try:
    from peft.utils import print_trainable_parameters
except Exception:
    def print_trainable_parameters(model):
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        print(f'Trainable params: {trainable_params} / {total_params} ({100*trainable_params/total_params:.2f}%)')

# RoBERTa/VulBERTa-style attention modules usually use query/value names.
# Keep auto-detection as a fallback in case your checkpoint uses different names.
def suggest_target_modules(model, max_candidates=8):
    candidates = []
    seen = set()
    for name, _module in model.named_modules():
        lname = name.lower()
        if any(k in lname for k in ['query', 'value', 'key', 'dense', 'fc1', 'fc2', 'q_proj', 'v_proj', 'out_proj']):
            part = name.split('.')[-1]
            if part not in seen:
                seen.add(part)
                candidates.append(part)
        if len(candidates) >= max_candidates:
            break
    return candidates

suggested = suggest_target_modules(model)
if suggested:
    TARGET_MODULES = suggested
else:
    TARGET_MODULES = ['query', 'value']

print('Using target modules for LoRA:', TARGET_MODULES)
print('Base model class:', model.__class__.__name__)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=TARGET_MODULES,
    lora_dropout=0.1,
    bias='none',
    task_type=TaskType.SEQ_CLS,
)

# If you want 8-bit preparation, uncomment the next line (and ensure bitsandbytes is installed)
# from peft import prepare_model_for_int8_training
# model = prepare_model_for_int8_training(model)

peft_model = get_peft_model(model, lora_config)

# Display trainable params using PEFT helper (or fallback)
print_trainable_parameters(peft_model)

Using target modules for LoRA: ['query', 'key', 'value', 'dense', 'out_proj']
Base model class: RobertaForSequenceClassification
Trainable params: 1919234 / 126566404 (1.52%)


## 7) Training Loop (using Trainer for brevity)

We use Trainer with a tiny training run for demonstration. For larger datasets, configure `TrainingArguments` accordingly.

In [36]:
from transformers import TrainingArguments, Trainer, EvalPrediction
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    learning_rate=2e-4,
    report_to=[]
)

def compute_metrics(p: EvalPrediction):
    predictions = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds = np.argmax(predictions, axis=-1)
    labels = p.label_ids
    average = 'binary' if len(np.unique(labels)) == 2 else 'macro'
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average=average, zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    compute_metrics=compute_metrics
)

print('Starting training (tiny demo)...')
train_result = trainer.train()
print('Training finished.')

metrics = trainer.evaluate()
print(metrics)

Starting training (tiny demo)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Training finished.


Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
No log,0.696674,1,0.000000,0.000000,0.000000,0.000000


{'eval_loss': 0.6966736316680908, 'eval_accuracy': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0}


## 8) Save Exported LoRA Adapters

Save only the LoRA adapter weights (small) so you can reload and use them without storing full base model weights.

In [37]:
ADAPTER_DIR = './lora_adapter'
peft_model.save_pretrained(ADAPTER_DIR)
print('LoRA adapter saved to', ADAPTER_DIR)

# Optionally copy to Drive if mounted
if DRIVE_AVAILABLE:
    import shutil
    dst = '/content/drive/MyDrive/lora_adapters/lora_adapter'
    shutil.copytree(ADAPTER_DIR, dst, dirs_exist_ok=True)
    print('Adapter copied to Drive:', dst)

LoRA adapter saved to ./lora_adapter
Adapter copied to Drive: /content/drive/MyDrive/lora_adapters/lora_adapter


## 9) Load LoRA-adapted Model and Inference

Reload base model and load saved LoRA adapter for inference on sample inputs.

In [38]:
from peft import PeftModel

# Load base model again (fresh) and apply adapter
base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model_with_adapter = PeftModel.from_pretrained(base, ADAPTER_DIR)
model_with_adapter.eval()

# Inference helper
import torch
from torch.nn.functional import softmax

def predict(texts):
    inputs = tokenizer(texts, truncation=True, padding=True, return_tensors='pt', max_length=MAX_LENGTH)
    if torch.cuda.is_available():
        model_with_adapter.to('cuda')
        inputs = {k:v.cuda() for k,v in inputs.items()}
    with torch.no_grad():
        logits = model_with_adapter(**inputs).logits
        probs = softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=1).cpu().numpy()
    return preds, probs.cpu().numpy()

sample_texts = [
    'Potential buffer overflow in copy routine.',
    'Helper function for formatting dates.'
]
print('Predictions:')
print(predict(sample_texts))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Predictions:
(array([1, 1]), array([[0.49845585, 0.5015441 ],
       [0.498707  , 0.50129294]], dtype=float32))


## 10) Visualize Training Metrics

Plot training loss and evaluation metrics saved by the Trainer.

In [39]:
import matplotlib.pyplot as plt

# Trainer logs may contain metrics; try to plot if present
log_history = getattr(trainer.state, 'log_history', [])
steps = [h.get('loss_step', i) for i,h in enumerate(log_history)]
losses = [h['loss'] for h in log_history if 'loss' in h]
if losses:
    plt.plot(losses)
    plt.title('Training loss (log history)')
    plt.xlabel('Logging step')
    plt.ylabel('Loss')
    plt.show()
else:
    print('No loss history available in trainer.state.log_history for this short demo.')

# Display sample predictions
preds, probs = predict(['Example vulnerable code', 'Clean helper function'])
print('Sample preds:', preds)
print('Sample probs:', probs)

No loss history available in trainer.state.log_history for this short demo.
Sample preds: [0 0]
Sample probs: [[0.50223285 0.49776718]
 [0.5042205  0.49577948]]
